In [2]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join('../..')))

In [3]:
from src.patient_data_dispatcher import PatientDataDispatcher
from src.core.enums import MileStone, DataFrequency, PatientDataType
from src.pipeline import dmo_for_random_forest
from src.model import DMORandomForestRegressor
from src.evaluation import Evaluation

import torch

In [4]:
dmo_features = [
    "wb_all_sum",
    "walkdur_all_sum",
    "wbsteps_all_sum",
    "wbdur_all_avg",
    "wbdur_all_p90",
    "wbdur_all_var",
    "cadence_all_avg",
    "strdur_all_avg",
    "cadence_all_var",
    "strdur_all_var",
    "ws_1030_avg",
    "strlen_1030_avg",
    "wb_10_sum",
    "ws_10_p90",
    "wb_30_sum",
    "ws_30_avg",
    "strlen_30_avg",
    "cadence_30_avg",
    "strdur_30_avg",
    "ws_30_p90",
    "cadence_30_p90",
    "ws_30_var",
    "strlen_30_var",
    "wb_60_sum",
    "total_worn_h",
]

In [5]:
static_features = [
    "weight",
    "height",
    "EDFSCR1L"
]

In [6]:
pdd = PatientDataDispatcher(
    "../../config/config.yaml",
    dmo_features,
    MileStone.ALL,
    data_frequency=DataFrequency.DAILY,
    physical_subset=True,
    #static_features=static_features
)
ids = list(set(pdd.metadata["Local.Participant"].to_list()))
dmo_data, dmo_labels = pdd.get_patient_data(PatientDataType.MILESTONE, ids=ids)

In [7]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# perform impute on a visit by visit basis, as some visits are completely missing
imputer = IterativeImputer(missing_values=-1, tol=1e-2, keep_empty_features=True)
patients, visits, features, days = dmo_data.shape

for p in range(patients):
    for v in range(visits):
        visit_data = dmo_data[p, v]

        if (visit_data == -1).all():
            continue

        dmo_data[p, v] = torch.from_numpy(imputer.fit_transform(visit_data)).to(dtype=torch.float64)

/home/gwilym-rutherford/Documents/Year 3 Tuos/dissertation/Workspace/Experiment 1/.venv/lib/python3.13/site-packages/sklearn/impute/_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


In [8]:
dmo_data.shape

torch.Size([597, 5, 7, 25])

In [14]:
dmo_labels.shape

torch.Size([597, 5, 1])

In [20]:
def get_patient_visits(dmo_data, dmo_labels, n_visits):
    """
    Filters patients who have at least n visits with both valid data and labels.
    Returns: (filtered_data, filtered_labels)
    """
    feat_valid = (dmo_data != -1).all(dim=-1).all(dim=-1)
    label_valid = (dmo_labels.squeeze(-1) != -1)

    combined_valid = feat_valid & label_valid

    visit_counts = combined_valid.sum(dim=1)
    patient_indices = (visit_counts >= n_visits).nonzero(as_tuple=True)[0]

    if len(patient_indices) == 0:
        return torch.empty(0), torch.empty(0)

    final_feats = []
    final_labels = []

    for idx in patient_indices:
        mask = combined_valid[idx] 
        
        valid_feats = dmo_data[idx][mask][:n_visits]
        valid_lbls = dmo_labels[idx][mask][:n_visits]

        final_feats.append(valid_feats)
        final_labels.append(valid_lbls)

    return torch.stack(final_feats), torch.stack(final_labels)

In [21]:
for i in range(1, 6):
    data, label = get_patient_visits(dmo_data, dmo_labels, n_visits=i)
    print(data.shape)

torch.Size([595, 1, 7, 25])
torch.Size([541, 2, 7, 25])
torch.Size([490, 3, 7, 25])
torch.Size([428, 4, 7, 25])
torch.Size([280, 5, 7, 25])
